In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = WebBaseLoader(urls).load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splitted_docs = text_splitter.split_documents(docs)

embd = OpenAIEmbeddings()

vector_store = FAISS.from_documents(
    splitted_docs,
    embd 
)

retriever = vector_store.as_retriever()


In [6]:
# Router
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class RouterQuery(BaseModel):
    """Route a user query to most relevant datasource."""
    datasource: Literal["vectorstore", "websearch"] = Field(description="Given a user question choose to route it websearch or vectorstore.")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_router = llm.with_structured_output(RouterQuery)

# prompt
system = """
You are an expert in routing a user question to a vectorstore or websearch.
The vectorstore conatins documents realted to agent, prompt engineering, and adversarial attack.
Use the vectorstore for questionson these topics. Otherwise, use websearch
"""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]
)

question_router = route_prompt | structured_llm_router


In [7]:
question_router.invoke({"question": "What are the types of agent memory?"})

c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouterQuery(datasource='vectorstore'), input_type=RouterQuery])
  return self.__pydantic_serializer__.to_python(


RouterQuery(datasource='vectorstore')

In [8]:
# Retriever Grader
class GradeDocuments(BaseModel):
    """Binary score on the relevant check on the retrieved document."""
    binary_score: str = Field(description="Documents are relevant to the question 'yes' or 'no'")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

sytem_prompt = """
You are a grader assessing relevance of the retrieved documents to a user question.
If the document conatins keyword(s) or semantic meaning related to user question, grade it as relevant.
It does not need to be a stringent test. The goal is filter out errorneous retrievals.
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
"""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sytem_prompt),
        ("human", "Retrieved documents: {documents}\n\n User question: {question}\n\n")
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader

In [69]:
## Hallucination Grader

# Data Model
class GradeHallucinations(BaseModel):
    """
    Binary score in hallucination present in generation answer.
    """
    binary_score: str = Field(description="Answer is grounded in the facts, 'yes' or 'no'")


# LLM 
llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_hallucination_grader = llm.with_structured_output(GradeHallucinations)

# prompt
system_prompt = """
You are a grader in assessing whether an LLM generation is grounded in / supported by a set of retrieved facts.
Give binary score 'yes' or 'no'. 'Yes' means that answer is grounded in / supported by the set of facts.
"""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Set of facts:\n\n {documents} \n\n LLM generation: {generation}")
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_hallucination_grader

In [70]:
# Answer Grader

# Data class
class AnswerGrader(BaseModel):
    """Binary score to assess answer addresses question."""
    binary_score: str = Field(description="Answer addresses the question, 'yes' or 'no'")

# llm with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_answer_grader = llm.with_structured_output(AnswerGrader)

# System Prompt
system_prompt = """
You are a grader assessing whether an answer addresses or resolves a question\n
Give a binary score 'yes' or 'no', where 'yes' means the answer resolves the question.
"""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sytem_prompt),
        ("human", "User Question: \n\n {question} \n\n LLM generation: {generation}")
    ]
)

answer_grader = answer_prompt | structured_llm_grader

In [71]:
### Question re-writer

## LLM 
llm = ChatOpenAI(model="gpt-40-mini", temperature=0)

# prompt
system_prompt = """
You are a question re-writer that converts an input question to a better version that is optimized\n
for vectorstore retrieval. Look at the input and try to to reason about the underlying semantic intent/meaning.
"""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Here is the initial question: \n {question} \n Formulate a improved question")
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()

In [72]:
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(k=5)

docs = web_search_tool.invoke({"query": "Current situation in Iran"})['results']
web_results = "\n".join([d["content"] for d in docs])

print(web_results)


The Strait of Hormuz is only closed to Iran's “enemies”, Mousavi said, adding that the US and Israel's war was at the “root of the current situation”.
Israel launched a widespread strike on Iran's world-largest gas field, triggering retaliation from Tehran against key energy sites across the Gulf.
Iran says US and Israel attacked Natanz nuclear facility ... A satellite image shows a closer view of the Natanz Nuclear Facility with new building damage.
11 hours ago. Trump and Iran trade threats over energy targets as war escalates · Iranian missile barrages struck residential buildings in Arad · 2 mins ago.
Iran strikes near Israeli nuclear research center as Trump threatens attacks on Iranian power plants · A look at who holds the reins of power in Iran since the


In [73]:
from typing import TypedDict, List

class GraphState(TypedDict):
    """
    Represents the state of Graph.

    Attributes:
        question: question
        generation: LLM generation
        documents: list of documents
    """
    question: str
    generation: str
    documents: List[str]

In [74]:
## Generate
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# chain
rag_chain = prompt | llm | StrOutputParser()

In [75]:
from langchain_core.documents import Document

# Nodes

def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state['question']
    documents = retriever.invoke(question)

    return {"documents": documents, "question": question}

def generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state['question']
    documents = state['documents']

    generation = rag_chain.invoke({"question": question, "context": documents})
    return {"documents": documents, "question": question, "generation": generation}

def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    # score each documents
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke(
            {"question": question, "documents": d.page_content}
        )

        grade = score.binary_score.lower()

        if grade == 'yes':
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            continue
    return {"documents": filtered_docs, "question": question}


def transform_query(state):
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]

    # re-writer
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}

def web_search(state):
    """
    Web search based on the re-phrased question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with appended web results
    """

    print("---WEB SEARCH---")
    question = state["question"]

    # web search
    docs = web_search_tool.invoke({"query": question})
    docs = docs['results']
    web_results = '\n'.join([d['content'] for d in docs])

    web_results = Document(page_content=web_results)
    return {"documents": web_results, "question": question}


# Edges

def route_question(state):
    """
    Route question to web search or RAG.

    Args:
        state (dict): The current graph state

    Returns:
        str: Next node to call
    """

    print("---ROUTE QUESTION---")
    question = state["question"]
    
    source = question_router.invoke({"question": question})

    # "vectorstore", "websearch"

    if source.datasource.lower() == 'websearch':
        print("---ROUTE QUESTION TO WEB SEARCH---")
        return "web_search"
    elif source.datasource.lower() == 'vectorstore':
        print("---ROUTE QUESTION TO RAG---")
        return "vectorstore"

    
def decide_to_generate(state):
    """
    Determines whether to generate an answer, or re-generate a question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """
    print("---ASSESS GRADED DOCUMENTS---")

    filtered_documents = state['documents']

    if not filtered_documents:
        print(
            "---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---"
        )
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "generate"

def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """
    print("---CHECK HALLUCINATIONS---")
    question = state['question']
    documents = state['documents']
    generation = state['generation']

    score = hallucination_grader.invoke(
        {"documents": documents, "generation": generation}
    )

    grade = score.binary_score.lower()

    if grade == 'yes':
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke(
            {"question": question, "generation": generation}
        )
        grade = score.binary_score.lower()
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"
        


In [76]:
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(GraphState)

workflow.add_node("web_search", web_search)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

workflow.add_conditional_edges(START, route_question, 
    {"web_search": "web_search", "vectorstore": "retrieve"})
workflow.add_edge("web_search", "generate")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", 
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate"
    }
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges("generate", grade_generation_v_documents_and_question,
    {
        "useful": END,
        "not useful": "transform_query",
        "not supported": "generate"   
    }
)

app = workflow.compile()

In [77]:
app.invoke({"question":"What is machine learning"})

---ROUTE QUESTION---


c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouterQuery(datasource='websearch'), input_type=RouterQuery])
  return self.__pydantic_serializer__.to_python(


---ROUTE QUESTION TO WEB SEARCH---
---WEB SEARCH---
---GENERATE---
---CHECK HALLUCINATIONS---


c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeHallucinations(binary_score='yes'), input_type=GradeHallucinations])
  return self.__pydantic_serializer__.to_python(


---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---


c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeDocuments(binary_score='yes'), input_type=GradeDocuments])
  return self.__pydantic_serializer__.to_python(


{'question': 'What is machine learning',
 'generation': 'Machine learning is a subset of artificial intelligence that focuses on developing algorithms that learn from data to make predictions or decisions without human intervention. It involves training models on datasets to recognize patterns and make accurate inferences about new data. Applications include tasks like image classification, data analysis, and predictive maintenance.',
 'documents': Document(metadata={}, page_content='Machine learning is the subset of artificial intelligence (AI) focused on algorithms that can “learn” the patterns of training data and, subsequently, make accurate\xa0*inferences*\xa0about new data. The central premise of machine learning (ML) is that if you optimize a model’s performance on a dataset of tasks that adequately resemble the real-world problems it will be used for—through a process called\xa0model training—the model can make accurate predictions on the\xa0new\xa0data it sees in its ultimate 

In [78]:
app.invoke({"question":"What is agent memory"})

---ROUTE QUESTION---


c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouterQuery(datasource='vectorstore'), input_type=RouterQuery])
  return self.__pydantic_serializer__.to_python(


---ROUTE QUESTION TO RAG---
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---


c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeDocuments(binary_score='no'), input_type=GradeDocuments])
  return self.__pydantic_serializer__.to_python(


---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
---GENERATE---
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---


{'question': 'What is agent memory',
 'generation': 'Agent memory in LLM-powered autonomous agents consists of short-term and long-term memory components. Short-term memory involves in-context learning, while long-term memory allows the agent to retain and recall information over extended periods, often using an external vector store for fast retrieval. This memory system enables agents to learn from past experiences and improve their performance over time.',
 'documents': [Document(id='06081b08-f78a-4cae-9f66-6061bc95715d', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerfu